In [1]:
import numpy as np # linear algebra
import polars as pl # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/march-machine-learning-mania-2025/Conferences.csv
/kaggle/input/march-machine-learning-mania-2025/SeedBenchmarkStage1.csv
/kaggle/input/march-machine-learning-mania-2025/WNCAATourneyDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2025/WRegularSeasonCompactResults.csv
/kaggle/input/march-machine-learning-mania-2025/MNCAATourneySeedRoundSlots.csv
/kaggle/input/march-machine-learning-mania-2025/MRegularSeasonDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2025/MNCAATourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2025/MGameCities.csv
/kaggle/input/march-machine-learning-mania-2025/WSecondaryTourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2025/WGameCities.csv
/kaggle/input/march-machine-learning-mania-2025/MSeasons.csv
/kaggle/input/march-machine-learning-mania-2025/WNCAATourneySlots.csv
/kaggle/input/march-machine-learning-mania-2025/MSecondaryTourneyTeams.csv
/kaggle/input/march-machine-learning-mania-20

In [2]:
import os
import shutil  # For handling directory removal (optional)

def delete_files_in_directory(directory_path, remove_empty_directory=False):
    """
    Deletes all files within the specified directory.

    Args:
        directory_path (str): The path to the directory.
        remove_empty_directory (bool, optional):  If True, attempts to remove the directory
                                              after deleting all files.  Defaults to False.
    """
    try:
        # Check if the directory exists
        if not os.path.isdir(directory_path):
            print(f"Error: Directory '{directory_path}' not found.")
            return

        # Iterate through all files and directories in the specified directory
        for filename in os.listdir(directory_path):
            file_path = os.path.join(directory_path, filename)

            try:
                if os.path.isfile(file_path):
                    os.remove(file_path)  # Delete the file
                    print(f"Deleted file: {file_path}")
                elif os.path.isdir(file_path):  # Handle subdirectories recursively
                    # Option 1: Delete all files in the subdirectory and then remove the subdirectory
                    delete_files_in_directory(file_path, remove_empty_directory=True)  # Recursively delete files
                    if remove_empty_directory:  # Only remove if the flag is set
                        try:
                            os.rmdir(file_path)  # Try to remove empty directory
                            print(f"Removed empty subdirectory: {file_path}")
                        except OSError as e:
                            print(f"Error removing subdirectory {file_path}: {e}")
                                # Option 2:  Completely delete the subdirectory and its contents
                    # shutil.rmtree(file_path) # Use this cautiously!  Deletes the directory and ALL contents.
                    # print(f"Deleted subdirectory (and contents): {file_path}")


            except Exception as e:
                print(f"Error deleting {file_path}: {e}")

        # Optionally remove the directory itself if it's now empty
        if remove_empty_directory:
            try:
                os.rmdir(directory_path)
                print(f"Removed empty directory: {directory_path}")
            except OSError as e:
                print(f"Error removing directory {directory_path}: {e}")

    except Exception as e:
        print(f"An error occurred: {e}")
                

delete_files_in_directory ("/kaggle/working/march_madness/models")
delete_files_in_directory ("/kaggle/working/march_madness/male/models")
delete_files_in_directory ("/kaggle/working/march_madness/female/models")

Error: Directory '/kaggle/working/march_madness/models' not found.
Error: Directory '/kaggle/working/march_madness/male/models' not found.
Error: Directory '/kaggle/working/march_madness/female/models' not found.


Wow, that is a lot of files 

This notebook will use only some information: 
The tournament seeds should have an impact on the results, so we use to indicate the strength for every team in the NCAA tournament for every season

WNCAATourneySeeds
MNCAATourneySeeds

to feed my model. We will manipulate the Data so it comes in a format that indicates the conference and the raking inside the conference for every season.  This will help the model to predict pervious tournaments (for training) and the 2025 season for test/submission.

The 2nd level are  the basic scores during the season

WRegularSeasonCompactResults
MRegularSeasonCompactResults

We will manipulate the Data so it comes in a format that indicates the season totals wins, loss, points with a weighted win for every season. this will help the model to predict pervious tournaments (for training) and the 2025 season for test/submission.

we keep male and female seperated 

there is one file that collects the stats for each team per season 

we then create a training file with all known tournament results and join this file with the teams season result file ; one time for the left team (first team), one time for the right (second team) 

The model trains on predicting past NCAA tournaments using the results of that season 


In [3]:
w_teams = pl.scan_csv("/kaggle/input/march-machine-learning-mania-2025/WTeams.csv").collect()
m_teams = pl.scan_csv("/kaggle/input/march-machine-learning-mania-2025/MTeams.csv").collect()
m_compact_results = pl.scan_csv("/kaggle/input/march-machine-learning-mania-2025/MRegularSeasonCompactResults.csv").collect()
w_compact_results = pl.scan_csv("/kaggle/input/march-machine-learning-mania-2025/WRegularSeasonCompactResults.csv").collect()
m_rankings = pl.scan_csv("/kaggle/input/march-machine-learning-mania-2025/MMasseyOrdinals.csv").collect()

w_seeds = pl.scan_csv("/kaggle/input/march-machine-learning-mania-2025/WNCAATourneySeeds.csv").collect()
m_seeds = pl.scan_csv("/kaggle/input/march-machine-learning-mania-2025/MNCAATourneySeeds.csv").collect()

m_NCAATourneyCompactResults = pl.scan_csv("/kaggle/input/march-machine-learning-mania-2025/MNCAATourneyCompactResults.csv").collect()
w_NCAATourneyCompactResults = pl.scan_csv("/kaggle/input/march-machine-learning-mania-2025/WNCAATourneyCompactResults.csv").collect()
# submission_stage1 = pl.scan_csv ("/kaggle/input/march-machine-learning-mania-2025/SampleSubmissionStage1.csv").collect()
submission_stage2 = pl.scan_csv ("/kaggle/input/march-machine-learning-mania-2025/SampleSubmissionStage2.csv").collect()

m_truth = pl.scan_csv ("/kaggle/input/march-machine-learning-mania-2025/MNCAATourneyCompactResults.csv").collect()
w_truth = pl.scan_csv ("/kaggle/input/march-machine-learning-mania-2025/WNCAATourneyCompactResults.csv").collect()


m_names = pl.scan_csv("/kaggle/input/march-machine-learning-mania-2025/MTeams.csv")
w_names = pl.scan_csv("/kaggle/input/march-machine-learning-mania-2025/WTeams.csv")

In [4]:
m_names.filter (pl.col("TeamName") =="Auburn").collect()


TeamID,TeamName,FirstD1Season,LastD1Season
i64,str,i64,i64
1120,"""Auburn""",1985,2025


In [5]:
def extract_seeds (seeds : pl.DataFrame) -> pl.DataFrame :
    
    result = seeds.with_columns (pl.col("Seed").str.head(1).alias("Conference"), 
                               pl.col("Seed").str.slice(1,2).cast(pl.UInt8).alias("C_Seed"))
    return result.drop("Seed")

female_seeds_Season = extract_seeds (w_seeds) 
male_seeds_Season = extract_seeds (m_seeds) 

In [6]:
# converting the historic NCAA results into the format used for submission, 

def create_submission_format (compact_NCAA_results : pl.DataFrame) -> pl.DataFrame :
    bare_minimum1 = compact_NCAA_results.select (["Season", "WTeamID", "LTeamID"]).filter (pl.col("WTeamID") < pl.col("LTeamID"))
    submission_format1 =  bare_minimum1.with_columns ((pl.col("Season").cast (pl.String) + "_" + 
                                               pl.col("WTeamID").cast (pl.String) + "_" + pl.col("LTeamID").cast (pl.String)).alias ('ID'), 
                                                     pl.lit(1).alias ('pred'))
    
    bare_minimum2 = compact_NCAA_results.select (["Season", "WTeamID", "LTeamID"]).filter (pl.col("WTeamID") > pl.col("LTeamID"))
    submission_format2 =  bare_minimum2.with_columns ((pl.col("Season").cast (pl.String) + "_" + 
                                               pl.col("LTeamID").cast (pl.String) + "_" + pl.col("WTeamID").cast (pl.String)).alias ('ID'), 
                                                     pl.lit(0).alias ('pred'))
    submission_format = pl.concat ([submission_format1,submission_format2  ]).sample (fraction = 1.0, shuffle = True)
    return submission_format.drop(["Season", "WTeamID", "LTeamID"]) 

historic_NCAA_m = create_submission_format (m_NCAATourneyCompactResults)
historic_NCAA_w = create_submission_format (w_NCAATourneyCompactResults)

historic_NCAA = pl.concat ([historic_NCAA_m, historic_NCAA_w])

display (historic_NCAA)

print  (f"NCAA men start :{historic_NCAA_m.get_column ('ID').min()}, end: {historic_NCAA_m.get_column ('ID').max()}")
print  (f"NCAA women start :{historic_NCAA_w.get_column ('ID').min()}, end: {historic_NCAA_w.get_column ('ID').max()}")

ID,pred
str,i32
"""2019_1211_1403""",0
"""1990_1412_1417""",0
"""2022_1104_1323""",0
"""2002_1227_1328""",0
"""2001_1417_1429""",1
…,…
"""2016_3120_3124""",0
"""2007_3331_3345""",0
"""2005_3277_3397""",1


NCAA men start :1985_1104_1112, end: 2024_1397_1400
NCAA women start :1998_3104_3256, end: 2024_3417_3465


In [7]:
# depreceated, as it does not add value to auto gluon 
def extract_rankings (rankings : pl.DataFrame) -> pl.DataFrame :
    rankings  = rankings.with_columns ((pl.col("Season").cast (pl.String) + "_" + pl.col("TeamID").cast(pl.String)).alias ("Season_Team"))

    last_ranking_day  = rankings.group_by(["Season", "SystemName"]).agg(pl.col("RankingDayNum").max()).sort("Season")

    rankings_last = rankings.join (last_ranking_day, how = "inner", on = ["Season", "SystemName", "RankingDayNum"])


    team_rankings = rankings_last.pivot("SystemName", index="Season_Team", values="OrdinalRank")

    team_rankings = team_rankings.with_columns (pl.col("Season_Team").str.split("_").list.first().cast (pl.Int64).alias ("Season"),
                                               pl.col("Season_Team").str.split("_").list.last().cast (pl.Int64).alias ("TeamID"))

    return team_rankings.drop("Season_Team")

male_team_rankings = extract_rankings (m_rankings)

print (male_team_rankings.head(3))

shape: (3, 195)
┌─────┬─────┬──────┬─────┬───┬──────┬──────┬────────┬────────┐
│ GC  ┆ MIC ┆ AP   ┆ ARG ┆ … ┆ PAC  ┆ WAB  ┆ Season ┆ TeamID │
│ --- ┆ --- ┆ ---  ┆ --- ┆   ┆ ---  ┆ ---  ┆ ---    ┆ ---    │
│ i64 ┆ i64 ┆ i64  ┆ i64 ┆   ┆ i64  ┆ i64  ┆ i64    ┆ i64    │
╞═════╪═════╪══════╪═════╪═══╪══════╪══════╪════════╪════════╡
│ 115 ┆ 130 ┆ null ┆ 141 ┆ … ┆ null ┆ null ┆ 2003   ┆ 1102   │
│ 164 ┆ 196 ┆ null ┆ 180 ┆ … ┆ null ┆ null ┆ 2003   ┆ 1103   │
│ 17  ┆ 43  ┆ null ┆ 37  ┆ … ┆ null ┆ null ┆ 2003   ┆ 1104   │
└─────┴─────┴──────┴─────┴───┴──────┴──────┴────────┴────────┘


In [8]:
display (w_compact_results.get_column("Season").unique().to_list())

[1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024,
 2025]

In [9]:
m_compact_results.filter (pl.col("Season") == 2021).describe()

statistic,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
str,f64,f64,f64,f64,f64,f64,str,f64
"""count""",3855.0,3855.0,3855.0,3855.0,3855.0,3855.0,"""3855""",3855.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0
"""mean""",2021.0,79.274189,1290.841245,76.808301,1288.496757,64.807004,null,0.067445
"""std""",0.0,31.420038,105.990924,10.516196,104.52898,10.284112,null,0.303271
"""min""",2021.0,23.0,1101.0,45.0,1101.0,30.0,"""A""",0.0
"""25%""",2021.0,51.0,1202.0,69.0,1199.0,58.0,null,0.0
"""50%""",2021.0,82.0,1288.0,76.0,1291.0,65.0,null,0.0
"""75%""",2021.0,107.0,1385.0,83.0,1379.0,71.0,null,0.0
"""max""",2021.0,132.0,1471.0,142.0,1471.0,120.0,"""N""",4.0


In [10]:
w_truth

Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
i64,i64,i64,i64,i64,i64,str,i64
1998,137,3104,94,3422,46,"""H""",0
1998,137,3112,75,3365,63,"""H""",0
1998,137,3163,93,3193,52,"""H""",0
1998,137,3198,59,3266,45,"""H""",0
1998,137,3203,74,3208,72,"""A""",0
…,…,…,…,…,…,…,…
2024,147,3163,80,3425,73,"""A""",0
2024,147,3234,94,3261,87,"""H""",0
2024,151,3234,71,3163,69,"""N""",0


In [11]:
def win_loss_record (teams : pl.DataFrame) -> pl.DataFrame :
    
    result = teams.select ("TeamID")
    base_file =  w_compact_results if (teams.get_column("TeamID").min() > 2000) else m_compact_results
    # season_results = base_file.filter (pl.col("Season") == season)
    # base_file = base_file.with_columns (pl.col("DayNum").max().over("Season").alias ("Season_length"))
    # p-day measures how late in the season a game is played with a min of 0.5 and a max of 1.5.this allows to give games 
    #    later in the season more importance 
    # Wmargin is the difference between win score and loose score 
    base_file = base_file.with_columns ((pl.col("DayNum")/ pl.col("DayNum").max().over("Season") + 0.5).alias ("p_day"), 
                                        (pl.col("WScore") - pl.col ("LScore")).alias ("WMargin"))  
        
    wins = base_file.group_by (["Season","WTeamID"]).len()
    losses = base_file.group_by (["Season","LTeamID"]).len()
    w_wins = base_file.group_by (["Season","WTeamID"]).agg (pl.col("p_day").sum() )
    w_losses = base_file.group_by (["Season","LTeamID"]).agg (pl.col("p_day").sum() )
    wscores =  base_file.group_by (["Season","WTeamID"]).agg (pl.col("WScore").sum())
    lscores = base_file.group_by (["Season","LTeamID"]).agg (pl.col("LScore").sum())
    opp_wscores = base_file.group_by (["Season","WTeamID"]).agg (pl.col("LScore").sum())
    opp_lscores = base_file.group_by (["Season","LTeamID"]).agg (pl.col("WScore").sum())
    wMargin = base_file.group_by (["Season","WTeamID"]).agg (pl.col("WMargin").mean())
    lMargin = base_file.group_by (["Season","LTeamID"]).agg (pl.col("WMargin").mean() * (-1))

    result = result.join (wins, how = "left", left_on = ["TeamID"], right_on = ["WTeamID"])
    result = result.rename ({"len" : "games won"})
    result = result.join (losses, how = "left", left_on = ["TeamID", "Season"], right_on = ["LTeamID", "Season"])
    result = result.rename ({"len" : "games lost"})
    result = result.join (w_wins, how = "left", left_on = ["TeamID", "Season"], right_on = ["WTeamID", "Season"])
    result = result.rename ({"p_day" : "weighted games won"})
    result = result.join (w_losses, how = "left", left_on = ["TeamID", "Season"], right_on = ["LTeamID", "Season"])
    result = result.rename ({"p_day" : "weighted games lost"}) 
    result = result.join (wscores, how = "left", left_on = ["TeamID", "Season"], right_on = ["WTeamID", "Season"])
    result = result.rename ({"WScore" : "points in win"})
    result = result.join (lscores, how = "left", left_on = ["TeamID", "Season"], right_on = ["LTeamID", "Season"])
    result = result.rename ({"LScore" : "points in loss"})
    result = result.join (opp_wscores, how = "left", left_on = ["TeamID", "Season"], right_on = ["WTeamID", "Season"])
    result = result.rename ({"LScore" : "dev points in win"})
    result = result.join (opp_lscores, how = "left", left_on = ["TeamID", "Season"], right_on = ["LTeamID", "Season"])
    result = result.rename ({"WScore" : "dev points in loss"})
    result = result.join (wMargin, how = "left", left_on = ["TeamID", "Season"], right_on = ["WTeamID", "Season"])
    result = result.rename ({"WMargin" : "mean margin win"})
    result = result.join (lMargin, how = "left", left_on = ["TeamID", "Season"], right_on = ["LTeamID", "Season"])
    result = result.rename ({"WMargin" : " mean margin loss"})
    
    result = result.with_columns ((pl.col("games won") + pl.col("games lost")).alias ("total games"), 
                                  (pl.col("points in win") + pl.col("points in loss")).alias ("total points"), 
                                  (pl.col("dev points in win") + pl.col("dev points in loss")).alias ("total dev points"))
    result = result.with_columns ((pl.col("total points") / pl.col("total games")).alias ("points mean"), 
                                  (pl.col("total dev points") / pl.col("total games")).alias ("dev points mean"), 
                                  (pl.col("dev points in win") + pl.col("dev points in loss")).alias ("total dev points"))
    
    just_games_won = result.select(["Season", "TeamID", "games won"]) 
    weight_score_per_game = base_file.join (just_games_won, how = "inner", left_on = ["Season","LTeamID"], right_on = ["Season", "TeamID"])
    weight_score = weight_score_per_game.group_by(["Season", "WTeamID"]).agg(pl.col("games won").sum())
    weight_score = weight_score.rename ({"games won" : "opponent games won"})
    result = result.join (weight_score, how = "left", left_on = ["TeamID", "Season"], right_on = ["WTeamID", "Season"])
        
    return result.fill_null(0)

male_team_results = win_loss_record (m_teams)

display (male_team_results)

female_team_results = win_loss_record (w_teams)

display (female_team_results)

TeamID,Season,games won,games lost,weighted games won,weighted games lost,points in win,points in loss,dev points in win,dev points in loss,mean margin win,mean margin loss,total games,total points,total dev points,points mean,dev points mean,opponent games won
i64,i64,u32,u32,f64,f64,i64,i64,i64,i64,f64,f64,u32,i64,i64,f64,f64,u32
1101,2024,14,17,15.007576,16.242424,1066,1134,958,1324,7.714286,-11.176471,31,2200,2282,70.967742,73.612903,178
1101,2023,9,17,8.606061,17.939394,709,1142,607,1340,11.333333,-11.647059,26,1851,1947,71.192308,74.884615,110
1101,2025,13,16,13.818182,15.916667,991,968,852,1207,10.692308,-14.9375,29,1959,2059,67.551724,71.0,154
1101,2020,16,11,18.6875,10.0,1220,745,1020,837,12.5,-8.363636,27,1965,1857,72.777778,68.777778,141
1101,2014,2,19,2.280303,18.545455,160,1166,153,1498,3.5,-17.473684,21,1326,1651,63.142857,78.619048,10
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1477,2023,12,20,12.416667,20.05303,864,1397,801,1574,5.25,-8.85,32,2261,2375,70.65625,74.21875,141
1478,2024,12,17,13.568182,15.772727,948,1081,754,1319,16.166667,-14.0,29,2029,2073,69.965517,71.482759,122
1478,2025,7,23,6.810606,22.469697,530,1628,484,1958,6.571429,-14.347826,30,2158,2442,71.933333,81.4,78


TeamID,Season,games won,games lost,weighted games won,weighted games lost,points in win,points in loss,dev points in win,dev points in loss,mean margin win,mean margin loss,total games,total points,total dev points,points mean,dev points mean,opponent games won
i64,i64,u32,u32,f64,f64,i64,i64,i64,i64,f64,f64,u32,i64,i64,f64,f64,u32
3101,2014,11,12,12.257576,11.863636,863,747,728,880,12.272727,-11.083333,23,1610,1608,70.0,69.913043,83
3101,2021,9,10,10.204545,12.113636,637,587,533,720,11.555556,-13.3,19,1224,1253,64.421053,65.947368,77
3101,2023,12,15,13.219697,15.44697,876,1010,694,1129,15.166667,-7.933333,27,1886,1823,69.851852,67.518519,120
3101,2025,17,12,17.060606,12.022727,1267,717,1010,849,15.117647,-11.0,29,1984,1859,68.413793,64.103448,178
3101,2022,13,13,13.068182,14.856061,999,845,808,1012,14.692308,-12.846154,26,1844,1820,70.923077,70.0,127
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3477,2025,5,22,5.386364,22.174242,342,1302,295,1654,9.4,-16.0,27,1644,1949,60.888889,72.185185,57
3478,2025,7,24,8.712121,22.590909,488,1182,416,1722,10.285714,-22.5,31,1670,2138,53.870968,68.967742,62
3478,2024,18,14,20.954545,11.295455,1151,674,918,1010,12.944444,-24.0,32,1825,1928,57.03125,60.25,170


In [12]:
def create_train (submission_format, tournament_results :pl.DataFrame) -> pl.DataFrame :
    
    truth_is_female = (tournament_results.get_column ("WTeamID").min() > 2000)
    
    result = submission_format.with_columns (pl.col("ID").str.split ("_").list.get(0).cast(pl.Int64).alias ("Season"),
                             pl.col("ID").str.split ("_").list.get(1).cast(pl.Int64).alias ("Team1"),
                             pl.col("ID").str.split ("_").list.get(2).cast(pl.Int64).alias ("Team2"))
   
    if truth_is_female :
        result = result.filter (pl.col('Team1') > 2000)
    else :
        result = result.filter (pl.col('Team1') < 2000)
   
    bare_minimum = tournament_results.select (["Season", "WTeamID", "LTeamID", "WScore", "LScore"])
    result = result.join (bare_minimum, how = "left", left_on = ["Season", "Team1", "Team2"], right_on = ["Season", "WTeamID", "LTeamID"])
    result = result.join (bare_minimum, how = "left", left_on = ["Season", "Team1", "Team2"], right_on = ["Season", "LTeamID", "WTeamID"])

    
    result = result.with_columns (pl.when (pl.col("WScore") > 0 ).then (pl.lit(1)).otherwise (
                                  pl.when (pl.col("WScore_right") > 0 ).then (pl.lit(0)).otherwise (
                                           pl.lit(0.5))).alias ("truth"))
    
    for unused in ['Pred', 'DayNum', 'WScore', 'LScore', 'WLoc', 'NumOT', 'DayNum_right', 
                   'WScore_right', 'LScore_right', 'WLoc_right', 'NumOT_right'] :
       if unused in result.columns :
           result = result.drop (unused)
            
    return result
    

In [13]:
male_training = create_train (historic_NCAA_m, m_truth).filter ((pl.col("truth") ==1) | (pl.col("truth") ==0))
female_training = create_train (historic_NCAA_w, w_truth).filter ((pl.col("truth") ==1) | (pl.col("truth") ==0))

male_submission = create_train (submission_stage2, m_truth)
female_submission = create_train (submission_stage2, w_truth)

print (f" total size {male_training.shape  }   ")
print (f" total size {male_submission.shape }   ")

print (male_training.head(3))

 total size (2518, 6)   
 total size (66066, 5)   
shape: (3, 6)
┌────────────────┬──────┬────────┬───────┬───────┬───────┐
│ ID             ┆ pred ┆ Season ┆ Team1 ┆ Team2 ┆ truth │
│ ---            ┆ ---  ┆ ---    ┆ ---   ┆ ---   ┆ ---   │
│ str            ┆ i32  ┆ i64    ┆ i64   ┆ i64   ┆ f64   │
╞════════════════╪══════╪════════╪═══════╪═══════╪═══════╡
│ 2019_1211_1403 ┆ 0    ┆ 2019   ┆ 1211  ┆ 1403  ┆ 0.0   │
│ 1990_1412_1417 ┆ 0    ┆ 1990   ┆ 1412  ┆ 1417  ┆ 0.0   │
│ 2022_1104_1323 ┆ 0    ┆ 2022   ┆ 1104  ┆ 1323  ┆ 0.0   │
└────────────────┴──────┴────────┴───────┴───────┴───────┘


In [14]:
def add_info (df, info : pl.DataFrame) -> pl.DataFrame :
  # the information in info for each team in df     
  df = df.join (info, how = "left", left_on = ["Season","Team1"], right_on = ["Season","TeamID"])
  df = df.join (info, how = "left", left_on = ["Season","Team2"], right_on = ["Season","TeamID"])  
  return df 

In [15]:
def build_pipeline (df : pl.DataFrame) -> pl.DataFrame :   
    if  df.get_column ("Team1").min() < 2000 :
        team_results = male_team_results
        seeds_season = male_seeds_Season
    else : 
        team_results = female_team_results
        seeds_season = female_seeds_Season
    return df.pipe (add_info, info = team_results).pipe (add_info, info = seeds_season)
    
    

In [16]:
male_training = build_pipeline (male_training)

try :
    my_weights = pl.scan_csv("/kaggle/working/mncaa_weights.csv").collect()
    male_training = male_training.join (my_weights, how = "right", on ="ID")
    print ("weights added for male NCAA training")
    print (f"final feature list for male NCAA  {male_training.columns = }")
except:
    print ("no weights available for male NCAA")

female_training = build_pipeline (female_training)
try :
    my_weights = pl.scan_csv("/kaggle/working/wncaa_weights.csv").collect()
    female_training = female_training.join (my_weights, how = "right", on ="ID")
    print ("weights added for female NCAA training")
    print (f"final feature list for female NCAA  {female_training.columns = }")
except:
    print ("no weights available for female NCAA")


male_submission = build_pipeline (male_submission)
female_submission = build_pipeline (female_submission)

no weights available for male NCAA
no weights available for female NCAA


In [17]:
!pip install ray==2.10.0
!pip install autogluon.tabular --no-cache-dir -q
!pip install -U ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.1/65.1 MB 24.1 MB/s eta 0:00:00
  Attempting uninstall: ray
    Found existing installation: ray 2.42.1
    Uninstalling ray-2.42.1:
      Successfully uninstalled ray-2.42.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.2/352.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.2/266.2 kB 246.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 213.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 177.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 202.7 MB/s eta 0:00:00


In [18]:
print (male_training.shape) 
print (male_submission.shape)
print (male_training.columns) 


(2518, 42)
(66066, 41)
['ID', 'pred', 'Season', 'Team1', 'Team2', 'truth', 'games won', 'games lost', 'weighted games won', 'weighted games lost', 'points in win', 'points in loss', 'dev points in win', 'dev points in loss', 'mean margin win', ' mean margin loss', 'total games', 'total points', 'total dev points', 'points mean', 'dev points mean', 'opponent games won', 'games won_right', 'games lost_right', 'weighted games won_right', 'weighted games lost_right', 'points in win_right', 'points in loss_right', 'dev points in win_right', 'dev points in loss_right', 'mean margin win_right', ' mean margin loss_right', 'total games_right', 'total points_right', 'total dev points_right', 'points mean_right', 'dev points mean_right', 'opponent games won_right', 'Conference', 'C_Seed', 'Conference_right', 'C_Seed_right']


In [19]:


from autogluon.tabular import TabularPredictor

if "my_weight" in male_training.columns :
    m_predictor = TabularPredictor(path = '/kaggle/working/march_madness/male',
                                       label='truth', 
                               problem_type = 'binary', 
                               eval_metric =  'accuracy',  
                               sample_weight = 'my_weight',
                               verbosity  = 2,
                               learner_kwargs = {'ignored_columns' : [
                                   'ID', "pred"]})
else :
    m_predictor = TabularPredictor(path = '/kaggle/working/march_madness/male',
                                       label='truth', 
                               problem_type = 'binary', 
                               eval_metric =  'accuracy',  
                               # sample_weight = 'my_weight',
                               verbosity  = 2,
                               learner_kwargs = {'ignored_columns' : [
                                   'ID', "pred"]})
m_predictor.fit(train_data= male_training.to_pandas(), 
                        presets= 'best_quality',
    # best_quality, high_quality, 'medium_quality, 'experimental_quality',                         
                        time_limit = 18000,
                        # num_gpus=0,
                        raise_on_no_models_fitted = True,
#                        dynamic_stacking=False, 
#                        num_bag_folds =2,
#                        num_stack_levels=1,
                        #hyperparameters=hyper_search,
#                         hyperparameters = my_search_hyperparameters  ,
                        #hyperparameter_tune_kwargs=hyperparameter_tune_kwargs,
                        )                              
                                

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Nov 10 10:07:59 UTC 2024
CPU Count:          4
Memory Avail:       29.96 GB / 31.35 GB (95.6%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. Then holdout validation data is used to detect stacked 

(_ray_fit pid=30050) [1000]	valid_set's binary_error: 0.278571


(_dystack pid=175) 	0.7324	 = Validation score   (accuracy)
(_dystack pid=175) 	28.9s	 = Training   runtime
(_dystack pid=175) 	0.14s	 = Validation runtime
(_dystack pid=175) Fitting model: NeuralNetFastAI_r143_BAG_L2 ... Training model for up to 181.61s of the 181.36s of remaining time.
(_dystack pid=175) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (4 workers, per: cpus=1, gpus=0, memory=0.05%)
(_ray_fit pid=30280) No improvement since epoch 4: early stopping
(_ray_fit pid=30443) No improvement since epoch 8: early stopping [repeated 4x across cluster]
(_dystack pid=175) 	0.7194	 = Validation score   (accuracy)
(_dystack pid=175) 	21.26s	 = Training   runtime
(_dystack pid=175) 	0.33s	 = Validation runtime
(_dystack pid=175) Fitting model: CatBoost_r70_BAG_L2 ... Training model for up to 156.83s of the 156.58s of remaining time.
(_dystack pid=175) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (4 workers,

# Leaderboard for male NCAA prediction

In [20]:
m_predictor.leaderboard() 

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost_BAG_L1,0.727164,accuracy,0.064043,31.236470,0.064043,31.236470,1,True,7
1,WeightedEnsemble_L2,0.727164,accuracy,0.065433,31.601637,0.001390,0.365167,2,True,111
2,NeuralNetFastAI_r187_BAG_L1,0.723193,accuracy,0.182662,23.026580,0.182662,23.026580,1,True,104
3,NeuralNetFastAI_r4_BAG_L1,0.721604,accuracy,0.243689,27.824422,0.243689,27.824422,1,True,98
4,CatBoost_r180_BAG_L1,0.721207,accuracy,0.067917,106.154236,0.067917,106.154236,1,True,89
...,...,...,...,...,...,...,...,...,...,...
106,RandomForest_r195_BAG_L1,0.678316,accuracy,0.205488,6.673804,0.205488,6.673804,1,True,26
107,RandomForest_r16_BAG_L1,0.673153,accuracy,0.379084,8.693300,0.379084,8.693300,1,True,94
108,RandomForest_r166_BAG_L1,0.672359,accuracy,0.198419,2.634176,0.198419,2.634176,1,True,76
109,KNeighborsDist_BAG_L1,0.611597,accuracy,0.015729,0.010672,0.015729,0.010672,1,True,2


In [21]:
historic_prediction = pl.Series (m_predictor.predict (male_training.to_pandas()))


male_weighted_train = male_training.with_columns (pl.when (pl.col("truth") == historic_prediction).then(
                                                      pl.lit(1)).otherwise (pl.lit(10)).alias ("my_weight")
)

print ("how many are predicted correctly")
print (male_weighted_train.group_by ("my_weight").len())

male_weighted_train = male_weighted_train.select (["ID", "my_weight"])

male_weighted_train.write_csv ("mncaa_weights.csv")


how many are predicted correctly
shape: (2, 2)
┌───────────┬──────┐
│ my_weight ┆ len  │
│ ---       ┆ ---  │
│ i32       ┆ u32  │
╞═══════════╪══════╡
│ 1         ┆ 2131 │
│ 10        ┆ 387  │
└───────────┴──────┘


In [22]:
print (female_training.shape) 
print (female_submission.shape)
print (female_training.columns) 


(1650, 42)
(65341, 41)
['ID', 'pred', 'Season', 'Team1', 'Team2', 'truth', 'games won', 'games lost', 'weighted games won', 'weighted games lost', 'points in win', 'points in loss', 'dev points in win', 'dev points in loss', 'mean margin win', ' mean margin loss', 'total games', 'total points', 'total dev points', 'points mean', 'dev points mean', 'opponent games won', 'games won_right', 'games lost_right', 'weighted games won_right', 'weighted games lost_right', 'points in win_right', 'points in loss_right', 'dev points in win_right', 'dev points in loss_right', 'mean margin win_right', ' mean margin loss_right', 'total games_right', 'total points_right', 'total dev points_right', 'points mean_right', 'dev points mean_right', 'opponent games won_right', 'Conference', 'C_Seed', 'Conference_right', 'C_Seed_right']


In [23]:

if "my_weight" in female_training.columns :
    w_predictor = TabularPredictor(path = '/kaggle/working/march_madness/female',
                                       label='truth', 
                               problem_type = 'binary', 
                               eval_metric =  'accuracy',  
                               sample_weight = 'my_weight',
                               verbosity  = 2,
                               learner_kwargs = {'ignored_columns' : [
                                   'ID', 'pred']})
else : 
    w_predictor = TabularPredictor(path = '/kaggle/working/march_madness/female',
                                       label='truth', 
                               problem_type = 'binary', 
                               eval_metric =  'accuracy',  
                               # sample_weight = 'my_weight',
                               verbosity  = 2,
                               learner_kwargs = {'ignored_columns' : [
                                   'ID', 'pred']})
    
w_predictor.fit(train_data= female_training.to_pandas(), 
                        presets= 'best_quality',
    # best_quality, high_quality, medium_quality, 'experimental_quality',                         
                        time_limit = 18000,
                        # num_gpus=0,
                        raise_on_no_models_fitted = True,
                        #dynamic_stacking=False, 
                        #num_stack_levels=1,
                        #hyperparameters=hyper_search,
#                         hyperparameters = my_search_hyperparameters  ,
                        #hyperparameter_tune_kwargs=hyperparameter_tune_kwargs,
                        )     

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Nov 10 10:07:59 UTC 2024
CPU Count:          4
Memory Avail:       29.33 GB / 31.35 GB (93.6%)
Disk Space Avail:   18.82 GB / 19.52 GB (96.4%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. Then holdout validation data is used to detect stacked 

# Leaderboard for female NCAA prediction

In [24]:
w_predictor.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost_r177_BAG_L2,0.833939,accuracy,2.658252,304.177607,0.080700,27.254121,2,True,123
1,WeightedEnsemble_L3,0.833939,accuracy,2.660087,304.515770,0.001835,0.338164,3,True,220
2,LightGBM_r135_BAG_L2,0.832727,accuracy,2.651015,291.522247,0.073463,14.598761,2,True,191
3,LightGBM_BAG_L2,0.832727,accuracy,2.664340,293.533710,0.086788,16.610224,2,True,113
4,LightGBM_r130_BAG_L2,0.832121,accuracy,2.661223,289.748334,0.083671,12.824849,2,True,140
...,...,...,...,...,...,...,...,...,...,...
215,LightGBMLarge_BAG_L1,0.766667,accuracy,0.086353,27.268806,0.086353,27.268806,1,True,13
216,RandomForestGini_BAG_L1,0.766667,accuracy,0.160304,1.451830,0.160304,1.451830,1,True,5
217,RandomForest_r16_BAG_L1,0.760000,accuracy,0.283355,5.366034,0.283355,5.366034,1,True,94
218,KNeighborsUnif_BAG_L1,0.661212,accuracy,0.008755,0.011949,0.008755,0.011949,1,True,1


In [25]:
historic_prediction = pl.Series (w_predictor.predict (female_training.to_pandas()))


female_weighted_train = female_training.with_columns (pl.when (pl.col("truth") == historic_prediction).then(
                                                      pl.lit(1)).otherwise (pl.lit(10)).alias ("my_weight")
)

print ("how many are predicted correctly")
print (female_weighted_train.group_by ("my_weight").len())

female_weighted_train = female_weighted_train.select (["ID", "my_weight"])

female_weighted_train.write_csv ("wncaa_weights.csv")

how many are predicted correctly
shape: (2, 2)
┌───────────┬──────┐
│ my_weight ┆ len  │
│ ---       ┆ ---  │
│ i32       ┆ u32  │
╞═══════════╪══════╡
│ 1         ┆ 1381 │
│ 10        ┆ 269  │
└───────────┴──────┘


In [26]:

male_submission = male_submission.fill_null(999.0)
male_submission = male_submission.fill_null("W")

male_march_madness_prediction =  m_predictor.predict_proba(male_submission.to_pandas()) 


In [27]:
female_submission = female_submission.fill_null(999.0)
female_submission = female_submission.fill_null("W")
female_march_madness_prediction =  w_predictor.predict_proba(female_submission.to_pandas())

In [28]:

march_madness_prediction = pl.concat ([pl.DataFrame(male_march_madness_prediction), pl.DataFrame(female_march_madness_prediction)], how = "vertical")

combined_id = pl.concat ([male_submission.get_column("ID"), female_submission.get_column("ID")], how = "vertical")

print (march_madness_prediction.head(3))

shape: (3, 2)
┌──────────┬──────────┐
│ 0        ┆ 1        │
│ ---      ┆ ---      │
│ f64      ┆ f64      │
╞══════════╪══════════╡
│ 0.542381 ┆ 0.457619 │
│ 0.591424 ┆ 0.408576 │
│ 0.769603 ┆ 0.230397 │
└──────────┴──────────┘


In [29]:
probabilty_first_team = march_madness_prediction.get_column ("1")
my_submit = pl.DataFrame ([combined_id, probabilty_first_team]) 
    
my_submit = my_submit.rename ({"1" : "Pred"})

my_submit.filter (pl.col("ID").str.contains ("1120"))



ID,Pred
str,f64
"""2025_1101_1120""",0.20642
"""2025_1102_1120""",0.214342
"""2025_1103_1120""",0.244955
"""2025_1104_1120""",0.438711
"""2025_1105_1120""",0.21078
…,…
"""2025_1120_1476""",0.774093
"""2025_1120_1477""",0.774707
"""2025_1120_1478""",0.77802


In [30]:
my_submit.write_csv("submission.csv")